In [1]:
import pandas as pd
from tqdm import tqdm
import os

In [2]:
#Read in accession lists for train, validation, and test sets
#Proces smaller training datasets as well (start with 100 genomes for hyperparameter tuning)
#train_accessions = open("../../data/processed_data/genome_partitions/train_partition_accessions_400_genomes.txt").read().splitlines()#[:2]
#train_accessions = open("../../data/processed_data/genome_partitions/train_partition_accessions_200_genomes.txt").read().splitlines()#[:2]
#train_accessions = open("../../data/processed_data/genome_partitions/train_partition_accessions_100_genomes.txt").read().splitlines()#[:2]

train_accessions = open("../../data/processed_data/genome_partitions/train_partition_accessions.txt").read().splitlines()
#val_accessions = open("../../data/processed_data/genome_partitions/val_partition_accessions.txt").read().splitlines()

#Define length of sequences
seqs_len = 300

In [3]:
def create_datasets_split(fold, 
                          archive, 
                          error_rates, 
                          model_type,
                          accession_filenames):
    """
    Create datasets for model development (train and validation splits).

    Args:
        fold (str): The fold type (train, val, test).
        archive (str): The archive type ("train_val" or "test").
        error_rates (str): Dataset with or without errors ("with_errors" or "without_errors").
        model_type (str): The type of model to create datasets for ("model_without_errors" or "model_with_errors").
        accession_filenames (list): List of accession filenames
    """

    #Initialize
    all_data_separate_rfs = []

    #Loop over accession filenames in fold
    for accession_filename in tqdm(accession_filenames):
        accession = accession_filename.strip(".csv")
        processed_reads = pd.read_csv(f"../../data/processed_data/reads_processed/{archive}/{error_rates}/csv/{accession_filename}.csv.gz", 
                                      compression='gzip',
                                      index_col=0)
        
        #Process each row of sequence data
        for _, row in processed_reads.iterrows():

            #Initialize indel detection per sequence
            indel_detected = False
            seq_errors = str(row["indel_positions"])

            if "I" in seq_errors or "D" in seq_errors:
                indel_detected = True
        
            #Make each RF into separate samples
            for frame in ["rf0", "rf1", "rf2"]:

                #Initialize
                desc = "unknown"

                #Format labels from string to list of integers
                labels = row[f"{frame}_labels"].replace(" ", "").replace("[", "").replace("]", "").split(",")
                labels = [int(label) for label in labels]

                #Label any indel transitions (instead of going between 0 and 1, they go from 0 -> 4 -> 1 or 1 -> 5 -> 0)
                if indel_detected:
                    if 1 in labels and 0 in labels:
                        # Find all transitions from 0 to 1 (mark as 4)
                        for i in range(1, len(labels)):
                            if labels[i] == 1 and labels[i-1] == 0:
                                labels[i] = 4
                        
                        # Find all transitions from 1 to 0 (mark the last 1 in each stretch as 5)
                        for i in range(len(labels) - 1):
                            if labels[i] == 1 and labels[i+1] == 0:
                                labels[i] = 5
                
                #Give each sequence a description based on labels to track loss 
                if set(labels) == {0}:
                    desc = "non-coding"
                
                #Only coding sequences
                elif set(labels) == {1}:
                    if str(row["MD:Z"]) != str(len(row["read"])):
                        desc = "coding_with_substitutions"
                    else:
                        desc = "coding"

                elif set(labels) == {0, 1, 2} or set(labels) == {0, 2} or set(labels) == {1, 2}:
                    desc = "transition_start"

                elif set(labels) == {0, 1, 3} or set(labels) == {1, 3} or set(labels) == {0, 3}:
                    desc = "transition_stop"

                elif set(labels) == {0, 1, 2, 3} or set(labels) == {1, 2, 3}:
                    desc = "transition_start_stop"
                
                elif 4 in labels or 5 in labels:
                    desc = "transition_indel"

                if frame == "rf0":
                    nt_seq = row["read"]
                elif frame == "rf1":
                    nt_seq = row["read"][1:] + "N" #Ensure stored nucleotide sequence has full length
                elif frame == "rf2":
                    nt_seq = row["read"][2:] + "NN" #Ensure stored nucleotide sequence has full length

                #if a description has been assigned, append the data
                #to the list of all data for separate RFs
                if desc != "unknown":
                    all_data_separate_rfs.append({
                    'accession': accession,
                    'nt_seq': nt_seq,
                    'aa_seq': row[f"{frame}_seq"],
                    'aa_labels': labels,
                    'aa_desc': desc})

                else:
                    print(f"AssertionError for {accession} in {frame} with sequencing errors {seq_errors} and labels {labels}")
    
    os.makedirs(f"../../data/processed_data/model_data/single_rf_crf/{model_type}/datasets_model", exist_ok=True)

    processed_samples_shared_df = pd.DataFrame(all_data_separate_rfs)
    processed_samples_shared_df.to_csv(f"../../data/processed_data/model_data/single_rf_crf/{model_type}/datasets_model/{fold}.csv.gz", index=False, compression="gzip")

# Create datasets with errors

In [ ]:
create_datasets_split(fold = "train", 
                      archive = "train_val", 
                      error_rates = "with_errors", 
                      model_type = "model_with_errors", 
                      accession_filenames = train_accessions)
"""
create_datasets_split(fold = "val", 
                      archive = "train_val", 
                      error_rates = "with_errors", 
                      model_type = "model_with_errors", 
                      accession_filenames = val_accessions)
"""

 89%|████████▊ | 721/813 [32:10<04:06,  2.68s/it]   


KeyboardInterrupt: 

: 

# Create datasets with no errors

In [5]:
"""
create_datasets_split(fold = "train", 
                      archive = "train_val", 
                      error_rates = "without_errors", 
                      model_type = "model_without_errors", 
                      accession_filenames = train_accessions)
create_datasets_split(fold = "val", 
                      archive = "train_val", 
                      error_rates = "without_errors", 
                      model_type = "model_without_errors", 
                      accession_filenames = val_accessions)
"""

'\ncreate_datasets_split(fold = "train", \n                      archive = "train_val", \n                      error_rates = "without_errors", \n                      model_type = "model_without_errors", \n                      accession_filenames = train_accessions)\ncreate_datasets_split(fold = "val", \n                      archive = "train_val", \n                      error_rates = "without_errors", \n                      model_type = "model_without_errors", \n                      accession_filenames = val_accessions)\n'